In [ ]:
# ============================================================
# 05 — LIGHTGBM RERANKER + Q4 HARNESS (MIND)
# Full-feature LightGBM reranker; AUC/MRR/nDCG@5/10 + bootstrap CIs + head/tail slices.
# Fully self-contained MIND notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers -q
import os, glob, re, math, time, zipfile, numpy as np, pandas as pd, datetime as dt, lightgbm as lgb, random, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
random.seed(0)
# ---- hardcoded MIND paths (small: train -> dev for offline metrics) ----
TRAIN = "/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train"
DEV   = "/kaggle/input/datasets/wrathofgod123/mind-dev/MINDsmall_dev"
SPLITS = [TRAIN, DEV]
NEWS = ["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH  = ["impression_id","user_id","time","history","impressions"]
_WORD = re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"

news = pd.concat([pd.read_csv(f"{d}/news.tsv", sep="\t", header=None, names=NEWS, quoting=3,
                 usecols=["news_id","category","title","abstract"]) for d in SPLITS]
                ).drop_duplicates("news_id").reset_index(drop=True)
news["title"] = news["title"].fillna(""); news["abstract"] = news["abstract"].fillna("")
cat_lut = {pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}
ids = [pfx(r.news_id) for r in news.itertuples()]
corpus = [tok(f"{r.title} {r.abstract}") for r in news.itertuples()]
id_to_row = {x:i for i,x in enumerate(ids)}
title_lut = {pfx(r.news_id):tok(r.title) for r in news.itertuples()}
print("articles:", len(ids))


In [ ]:
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b;s.N=len(c);s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float);s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r];dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg);v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
    def scores_all(s,q):
        return np.array([s.score(q,r) for r in range(s.N)])
bm25 = BM25(corpus); print("BM25 built")

from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # English-specialised
emb_txt = [f"{r.title} {r.abstract}".strip() for r in news.itertuples()]
emb_mat = minilm.encode(emb_txt, batch_size=512, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)
emb_by_id = {ids[i]:emb_mat[i] for i in range(len(ids))}
print("MiniLM encoded:", emb_mat.shape)


In [ ]:
def load_beh(p):
    b = pd.read_csv(f"{p}/behaviors.tsv", sep="\t", header=None, names=BEH, quoting=3)
    b["t"] = pd.to_datetime(b["time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"); return b

hist_lut = {}
first_seen = {}
click_ev = defaultdict(list); imp_ev = defaultdict(list)
_behs = [load_beh(p) for p in SPLITS]
for b in _behs:
    for u,h in zip(b["user_id"], b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)] = [pfx(x) for x in h.split()]
for b in _behs:
    for t,imps in zip(b["t"], b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tk in imps.split():
            nid = pfx(tk.split("-")[0])
            if nid not in first_seen or t < first_seen[nid]: first_seen[nid] = t
# click/impression events only from labeled splits (train+dev), not unlabeled test
for b in _behs[:2]:
    for t,imps in zip(b["t"], b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tk in imps.split():
            p = tk.split("-")
            if len(p)==2:
                nid = pfx(p[0]); imp_ev[nid].append(t)
                if p[1]=="1": click_ev[nid].append(t)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()

def hist_q(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];q=[]
    for x in ai: q.extend(title_lut.get(x,[]))
    return q
def hist_vecs(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];return [emb_by_id[x] for x in ai if x in emb_by_id]
def recency(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T);return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))
def ctr(aid,T,w=None):
    c=cnt(click_ev,aid,T,w);s=cnt(imp_ev,aid,T,w);return c/s if s>0 else 0.0
def user_cats(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];cats=[cat_lut.get(x) for x in ai]
    tot=len([c for c in cats if c]);cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
print("behavioural feature functions ready")

b_dv = load_beh(DEV)
print("dev impressions:", len(b_dv))
FEAT = ["bm25","emb_mean","emb_best","recency","pop_1h","pop_24h","pop_7d","pop_vel",
        "ctr_24h","ctr_total","cat_aff","position","slate_size"]
def feats(uid,T,cand):
    q=hist_q(uid);hv=hist_vecs(uid)
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    uc=user_cats(uid);m=len(cand);F=[]
    for i,c in enumerate(cand):
        bm=bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0
        cv=emb_by_id.get(c)
        em=float(um@cv) if (um is not None and cv is not None) else 0.0
        eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
        rec=recency(c,T)
        p1=cnt(click_ev,c,T,1);p24=cnt(click_ev,c,T,24);p7=cnt(click_ev,c,T,168)
        pv=p1/(p24+1);c24=ctr(c,T,24);ct=ctr(c,T)
        ca=uc.get(cat_lut.get(c),0.0);pos=i/max(1,m-1)
        F.append([bm,em,eb,rec,p1,p24,p7,pv,c24,ct,ca,pos,m])
    F=np.array(F)
    for col in [0,1,2,3,4,5,6,7]: F[:,col]=mm(F[:,col])
    return F
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def mrr_i(s,lb):
    o=np.argsort(-s)
    for rank,idx in enumerate(o,1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def ndcg_i(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    ideal=np.sort(lb)[::-1][:k]
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(ideal))
    return float(dcg/idcg) if idcg>0 else 0.0
def iter_impressions(b_df):
    """yield (uid, T, candidates, labels) for impressions with a click + history."""
    for u,t,imps in zip(b_df["user_id"], b_df["t"], b_df["impressions"]):
        if not isinstance(imps,str) or pd.isna(t): continue
        cand=[];labs=[]
        for tk in imps.split():
            p=tk.split("-")
            if len(p)==2: cand.append(pfx(p[0])); labs.append(int(p[1]))
        if cand and sum(labs)>0:
            yield pfx(u), t, cand, np.array(labs)


In [ ]:
# ---- build train (from TRAIN) and dev feature matrices ----
b_tr = load_beh(TRAIN)
def build(b_df, limit=None):
    rows=list(iter_impressions(b_df))
    if limit:
        import random as _r; _r.seed(0); _r.shuffle(rows); rows=rows[:limit]
    X=[];y=[];g=[]
    for uid,T,cand,labs in rows:
        X.append(feats(uid,T,cand)); y.append(labs); g.append(len(cand))
    return np.vstack(X), np.concatenate(y), g
print("building train features...")
Xtr,ytr,gtr = build(b_tr)
print("building dev features...")
Xdv,ydv,gdv = build(b_dv)
print("train:", Xtr.shape, "dev:", Xdv.shape)


In [ ]:
# ---- train LightGBM LambdaMART ----
rk = lgb.LGBMRanker(objective="lambdarank", n_estimators=500, learning_rate=0.03,
                    num_leaves=31, min_child_samples=50, importance_type="gain", verbose=-1)
rk.fit(Xtr, ytr, group=gtr)
print("feature importances (gain):")
for f,i in sorted(zip(FEAT, rk.feature_importances_), key=lambda z:-z[1]):
    print(f"  {f:12s} {i:>10.0f}")


In [ ]:
# ---- Q4 metrics with bootstrap 95% CIs + head/tail slices ----
# head/tail by candidate popularity: an impression is "head" if its clicked article is popular
def slice_of(cand, labs, T):
    ci=int(np.argmax(labs)); pop=cnt(click_ev,cand[ci],T,24)
    return "head" if pop>=5 else "tail"

sc = rk.predict(Xdv)
rows=list(iter_impressions(b_dv))
pos=0; per={"all":{"auc":[],"mrr":[],"n5":[],"n10":[]},
            "head":{"auc":[],"mrr":[],"n5":[],"n10":[]},
            "tail":{"auc":[],"mrr":[],"n5":[],"n10":[]}}
for (uid,T,cand,labs),gi in zip(rows,gdv):
    s=sc[pos:pos+gi]; pos+=gi
    a=auc_i(s,labs); mr=mrr_i(s,labs); n5=ndcg_i(s,labs,5); n10=ndcg_i(s,labs,10)
    grp=slice_of(cand,labs,T)
    for key in ("all",grp):
        if a is not None: per[key]["auc"].append(a)
        per[key]["mrr"].append(mr); per[key]["n5"].append(n5); per[key]["n10"].append(n10)

def ci(vals,nb=500):
    v=np.array([x for x in vals if x is not None]); 
    if len(v)==0: return (0,0,0)
    rng=np.random.default_rng(0)
    means=[rng.choice(v,len(v),replace=True).mean() for _ in range(nb)]
    return v.mean(), np.percentile(means,2.5), np.percentile(means,97.5)

print("=== MIND LightGBM Q4 metrics ===")
for grp in ("all","head","tail"):
    m,lo,hi=ci(per[grp]["auc"])
    print(f"[{grp}] AUC {m:.4f} (95% CI {lo:.4f}-{hi:.4f}) | "
          f"MRR {np.mean(per[grp]['mrr']):.4f} | "
          f"nDCG@5 {np.mean(per[grp]['n5']):.4f} | nDCG@10 {np.mean(per[grp]['n10']):.4f} "
          f"(n={len(per[grp]['mrr'])})")
